# Using the MethodRegistry Class in baseobjects

## Introduction

The `MethodRegistry` class provides a specialized registry for storing and managing methods. It extends the `FunctionRegistry` class to create a dictionary-like object specifically designed for holding callable methods that can be bound to objects. This makes it easy to organize, access, and manage a collection of methods that maintain their binding context when accessed.

This tutorial will guide you through:
- Understanding the purpose and functionality of the `MethodRegistry` class
- Creating and using a method registry
- Understanding how method binding works
- Using the descriptor protocol for automatic binding
- Practical use cases for method registries

**Prerequisites:**
- Basic understanding of Python dictionaries
- Familiarity with Python's callable objects
- Understanding of methods and the difference between functions and methods
- Basic knowledge of Python's descriptor protocol

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

In [7]:
from baseobjects.functions import MethodRegistry

## Core Functionality

The `MethodRegistry` class is designed to store and manage callable methods in a dictionary-like structure, with a special focus on maintaining the binding between methods and their instances. It extends the functionality of `FunctionRegistry` by adding the ability to automatically bind methods to instances when accessed.

### Basic Concept

At its core, `MethodRegistry` is a specialized registry that:

1. Stores callable objects (functions, methods) like `FunctionRegistry`
2. Implements the descriptor protocol to bind methods to instances
3. Returns a `BoundMethodRegistry` when accessed through an instance
4. Maintains the binding context of methods

Let's start with a simple example to understand how `MethodRegistry` works:

In [8]:
# Define a simple class with methods
class Calculator:
    # Define a MethodRegistry as a class attribute
    methods = MethodRegistry()

    def __init__(self, name) -> None:
        self.name = name

    def add(self, a, b):
        return a + b

    def subtract(self, a, b):
        return a - b

    def multiply(self, a, b):
        return a * b

    def divide(self, a, b):
        if b == 0:
            msg = "Cannot divide by zero"
            raise ValueError(msg)
        return a / b

    def get_name(self) -> str:
        return f"Calculator: {self.name}"


# Register the methods with the MethodRegistry
Calculator.methods.update_from_object(Calculator)

# Create calculator instances
calc1 = Calculator("Basic")
calc2 = Calculator("Scientific")

# Print the methods in the registry
print("Methods in registry:", list(calc1.methods.keys()))

Methods in registry: ['__hash__', '__init_subclass__', '__getattribute__', '__le__', '__lt__', 'add', '__ne__', '__reduce__', 'subtract', '__dir__', 'multiply', '__ge__', '__class__', '__reduce_ex__', 'get_name', '__subclasshook__', '__setattr__', '__new__', '__sizeof__', '__delattr__', 'divide', '__gt__', '__repr__', '__format__', '__str__', '__init__', '__eq__', '__getstate__']


Now let's see how to use the methods from the registry:

In [9]:
# Use methods from calc1's registry
print(f"calc1.methods['add'](5, 3): {calc1.methods['add'](5, 3)}")
print(f"calc1.methods['subtract'](10, 4): {calc1.methods['subtract'](10, 4)}")
print(f"calc1.methods['get_name'](): {calc1.methods['get_name']()}")

# Use methods from calc2's registry
print(f"\ncalc2.methods['multiply'](7, 6): {calc2.methods['multiply'](7, 6)}")
print(f"calc2.methods['divide'](20, 5): {calc2.methods['divide'](20, 5)}")
print(f"calc2.methods['get_name'](): {calc2.methods['get_name']()}")

calc1.methods['add'](5, 3): 8
calc1.methods['subtract'](10, 4): 6
calc1.methods['get_name'](): Calculator: Basic

calc2.methods['multiply'](7, 6): 42
calc2.methods['divide'](20, 5): 4.0
calc2.methods['get_name'](): Calculator: Scientific


### Comparing FunctionRegistry and MethodRegistry

To understand the difference between `FunctionRegistry` and `MethodRegistry`, let's compare how they handle methods:

In [10]:
# Import the necessary classes
from baseobjects.functions import FunctionRegistry

# Create a FunctionRegistry and add methods from Calculator
function_registry = FunctionRegistry()
function_registry.update_from_object(Calculator)

# Try to use a method from the FunctionRegistry
try:
    # This will fail because the method is not bound to an instance
    result = function_registry["get_name"]()
    print(result)
except TypeError as e:
    print(f"Error with FunctionRegistry: {e}")

    # We need to pass an instance explicitly
    result = function_registry["get_name"](calc1)
    print(f"With explicit instance: {result}")

# Now use a method from the MethodRegistry
print("\nWith MethodRegistry (automatically bound):")
print(f"calc1.methods['get_name'](): {calc1.methods['get_name']()}")
print(f"calc2.methods['get_name'](): {calc2.methods['get_name']()}")

Error with FunctionRegistry: Calculator.get_name() missing 1 required positional argument: 'self'
With explicit instance: Calculator: Basic

With MethodRegistry (automatically bound):
calc1.methods['get_name'](): Calculator: Basic
calc2.methods['get_name'](): Calculator: Scientific


As you can see, when using `FunctionRegistry`, we need to explicitly pass the instance as the first argument to the method. But with `MethodRegistry`, the methods are automatically bound to the instance through which we access the registry, so we don't need to pass the instance explicitly.

### Understanding Method Binding

In Python, methods are functions that are bound to an instance of a class. When you call a method on an instance, the instance is automatically passed as the first argument (`self`). The `MethodRegistry` class leverages Python's descriptor protocol to achieve this binding behavior.

Let's explore how this works in more detail:

In [11]:
# Get the class attribute (the MethodRegistry)
registry = Calculator.methods

# Manually create bound registries
bound_registry1 = registry.__get__(calc1, Calculator)
bound_registry2 = registry.__get__(calc2, Calculator)

# Verify that these are BoundMethodRegistry instances
print(f"calc1.methods is a {type(calc1.methods).__name__}")
print(f"bound_registry1 is a {type(bound_registry1).__name__}")
print(f"Is calc1.methods the same object as bound_registry1? {calc1.methods is bound_registry1}")

# Show that the bound registries are bound to different instances
print(f"\nbound_registry1['get_name'](): {bound_registry1['get_name']()}")
print(f"bound_registry2['get_name'](): {bound_registry2['get_name']()}")

# We can also create a bound registry for a new instance without adding it as an attribute
calc3 = Calculator("Advanced")
bound_registry3 = registry.__get__(calc3, Calculator)
print(f"\nbound_registry3['get_name'](): {bound_registry3['get_name']()}")

calc1.methods is a BoundMethodRegistry
bound_registry1 is a BoundMethodRegistry
Is calc1.methods the same object as bound_registry1? False

bound_registry1['get_name'](): Calculator: Basic
bound_registry2['get_name'](): Calculator: Scientific

bound_registry3['get_name'](): Calculator: Advanced


As you can see, when we access `methods` through an instance (e.g., `calc1.methods`), the descriptor protocol (`__get__` method) is triggered, which returns a `BoundMethodRegistry` bound to that instance. This allows us to call methods from the registry without explicitly passing the instance.

### Creating a Registry with Initial Methods

You can also create a `MethodRegistry` with initial methods by passing them to the constructor. Let's create a class that uses a MethodRegistry initialized with methods:

In [12]:
# Create a class with a MethodRegistry initialized with methods
class TextProcessor:
    def __init__(self, name) -> None:
        self.name = name

    def uppercase(self, text):
        return text.upper()

    def lowercase(self, text):
        return text.lower()

    def capitalize(self, text):
        return text.capitalize()

    def get_processor_name(self) -> str:
        return f"Text processor: {self.name}"


# Create a class with a MethodRegistry initialized with methods from TextProcessor
class EnhancedTextProcessor:
    # Create a MethodRegistry as a class attribute with initial methods
    processors = MethodRegistry()

    def __init__(self, name, prefix="") -> None:
        self.name = name
        self.prefix = prefix

    def add_prefix(self, text):
        return self.prefix + text

    def remove_prefix(self, text):
        if text.startswith(self.prefix):
            return text[len(self.prefix) :]
        return text

    def get_processor_info(self) -> str:
        return f"Enhanced processor: {self.name} (prefix: '{self.prefix}')"


# Add methods from TextProcessor to the EnhancedTextProcessor.processors registry
EnhancedTextProcessor.processors.update_from_object(TextProcessor)

# Add methods from EnhancedTextProcessor to its own registry
EnhancedTextProcessor.processors.update_from_object(EnhancedTextProcessor)

# Create instances
processor1 = EnhancedTextProcessor("Standard", ">> ")
processor2 = EnhancedTextProcessor("Advanced", "** ")

# Create a TextProcessor instance to demonstrate binding
text_processor = TextProcessor("Basic")

# Check what methods are in the registry
print("Methods in registry:", list(processor1.processors.keys()))

# Use methods from TextProcessor through processor1
print("\nUsing processor1.processors:")
print(f"uppercase: {processor1.processors['uppercase']('hello world')}")
print(f"lowercase: {processor1.processors['lowercase']('Hello World')}")
print(f"get_processor_name: {processor1.processors['get_processor_name']()}")

# Use methods from EnhancedTextProcessor through processor1
print("\nUsing processor1's own methods:")
print(f"add_prefix: {processor1.processors['add_prefix']('hello world')}")
print(f"get_processor_info: {processor1.processors['get_processor_info']()}")

# Use methods from EnhancedTextProcessor through processor2
print("\nUsing processor2's methods:")
print(f"add_prefix: {processor2.processors['add_prefix']('hello world')}")
print(f"get_processor_info: {processor2.processors['get_processor_info']()}")

Methods in registry: ['lowercase', '__hash__', '__init_subclass__', '__getattribute__', '__le__', '__lt__', '__ne__', '__reduce__', '__dir__', 'uppercase', 'capitalize', '__ge__', '__class__', '__reduce_ex__', '__subclasshook__', '__setattr__', '__sizeof__', '__new__', '__gt__', '__repr__', '__format__', 'get_processor_name', '__str__', '__init__', '__eq__', '__delattr__', '__getstate__', 'add_prefix', 'remove_prefix', 'get_processor_info']

Using processor1.processors:
uppercase: HELLO WORLD
lowercase: hello world
get_processor_name: Text processor: Standard

Using processor1's own methods:
add_prefix: >> hello world
get_processor_info: Enhanced processor: Standard (prefix: '>> ')

Using processor2's methods:
add_prefix: ** hello world
get_processor_info: Enhanced processor: Advanced (prefix: '** ')


## Module Interaction

The `MethodRegistry` class interacts with several other components of the baseobjects package:

1. It extends `BaseMethodRegistry`, which in turn extends `FunctionRegistry` from the `baseobjects.functions` module
2. It uses `BaseReducible` from the `baseobjects.bases` module
3. It leverages the `AnyCallable` type from `baseobjects.typing` for type hints
4. It works with `BoundMethodRegistry` for binding methods to instances

Let's explore how `MethodRegistry` interacts with these components:

In [13]:
# Create a custom registry that extends MethodRegistry
class FilteredMethodRegistry(MethodRegistry):
    """A method registry that only accepts methods with specific parameter types."""

    def __init__(self, param_type=None, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.param_type = param_type

    def update_from_object(self, object_) -> None:
        """Override to filter methods based on parameter type."""
        # Get all callable attributes
        for name in set(dir(object_)) | set(vars(object_).keys()):
            attr = getattr(object_, name, None)
            if attr is None or not callable(attr):
                continue

            # Check if the method has annotations
            if hasattr(attr, "__annotations__"):
                # Check if any parameter has the specified type
                has_param_type = False
                for param_name, param_type in attr.__annotations__.items():
                    if param_name != "return" and param_type == self.param_type:
                        has_param_type = True
                        break

                # Only add methods with the specified parameter type
                if self.param_type is None or has_param_type:
                    self.data[name] = attr.__func__ if hasattr(attr, "__func__") else attr
            else:
                # If no annotations, add only if no type filter is specified
                if self.param_type is None:
                    self.data[name] = attr.__func__ if hasattr(attr, "__func__") else attr


# Create a class with annotated methods
class DataProcessor:
    def __init__(self, name) -> None:
        self.name = name

    def process_string(self, text: str) -> str:
        return f"Processed string: {text.upper()}"

    def process_number(self, number: int) -> str:
        return f"Processed number: {number * 2}"

    def process_list(self, items: list) -> list:
        return sorted(items)

    def get_processor_name(self) -> str:
        return f"Processor: {self.name}"


# Create a processor instance
processor = DataProcessor("Main Processor")

# Create a registry that only accepts methods with string parameters
string_registry = FilteredMethodRegistry(param_type=str)
string_registry.update_from_object(processor)

# Create a registry that only accepts methods with int parameters
int_registry = FilteredMethodRegistry(param_type=int)
int_registry.update_from_object(processor)

# Create a registry that accepts all methods
all_registry = FilteredMethodRegistry()
all_registry.update_from_object(processor)

# Create bound registries
bound_string_registry = string_registry.__get__(processor)
bound_int_registry = int_registry.__get__(processor)
bound_all_registry = all_registry.__get__(processor)

# Check what methods are in each registry
print("String methods:", list(bound_string_registry.keys()))
print("Int methods:", list(bound_int_registry.keys()))
print("All methods:", list(bound_all_registry.keys()))

# Use the methods
print(f"\nString method: {bound_string_registry['process_string']('hello')}")
print(f"Int method: {bound_int_registry['process_number'](42)}")
print(f"List method: {bound_all_registry['process_list']([3, 1, 4, 1, 5, 9])}")

String methods: ['process_string']
Int methods: ['process_number']
All methods: ['__hash__', '__init_subclass__', '__getattribute__', '__le__', '__lt__', '__ne__', 'process_number', 'process_list', '__reduce__', '__dir__', 'process_string', '__ge__', '__class__', '__reduce_ex__', '__subclasshook__', '__setattr__', '__new__', '__sizeof__', '__delattr__', '__gt__', '__repr__', '__format__', 'get_processor_name', '__str__', '__init__', '__eq__', '__getstate__']

String method: Processed string: HELLO
Int method: Processed number: 84
List method: [1, 1, 3, 4, 5, 9]


### Integration with BaseReducible

The `BaseMethodRegistry` class inherits from `BaseReducible`, which provides functionality for pickling and unpickling objects. This is particularly important for `BoundMethodRegistry`, which needs to handle weak references to bound instances during pickling.

Let's see how this works:

In [14]:
import pickle


# Create a class with a method registry
class SerializableProcessor:
    operations = MethodRegistry()

    def __init__(self, name) -> None:
        self.name = name

    def process(self, data) -> str:
        return f"{self.name} processed: {data}"

    def get_info(self) -> str:
        return f"Processor: {self.name}"


# Add methods to the registry
SerializableProcessor.operations.update_from_object(SerializableProcessor)

# Create an instance
processor = SerializableProcessor("Pickle Test")

# Use the registry
print(f"Before pickling: {processor.operations['get_info']()}")

# Pickle and unpickle the instance
pickled_data = pickle.dumps(processor)
unpickled_processor = pickle.loads(pickled_data)

# Use the registry after unpickling
print(f"After unpickling: {unpickled_processor.operations['get_info']()}")

# Verify that the method is still bound to the correct instance
print(f"Processing data: {unpickled_processor.operations['process']('test data')}")

Before pickling: Processor: Pickle Test
After unpickling: Processor: Pickle Test
Processing data: Pickle Test processed: test data


This example demonstrates that the `MethodRegistry` and `BoundMethodRegistry` classes properly handle serialization and deserialization, maintaining the binding between methods and their instances even after pickling and unpickling.

## Advanced Features

### Dynamic Method Selection with MethodRegistry

One advanced use case for `MethodRegistry` is implementing dynamic method selection based on runtime conditions. This is similar to the Strategy pattern but with automatic binding of methods to instances:

In [15]:
# Create a class with different text processing strategies
class TextProcessor:
    # Create a method registry as a class attribute
    strategies = MethodRegistry()

    def __init__(self, default_strategy="uppercase") -> None:
        self.default_strategy = default_strategy
        self.text_cache = {}

    def uppercase(self, text):
        """Convert text to uppercase."""
        return text.upper()

    def lowercase(self, text):
        """Convert text to lowercase."""
        return text.lower()

    def capitalize(self, text):
        """Capitalize the first letter of each word."""
        return " ".join(word.capitalize() for word in text.split())

    def reverse(self, text):
        """Reverse the text."""
        return text[::-1]

    def process(self, text, strategy=None):
        """Process text using the specified strategy or default strategy."""
        # Use the specified strategy or the default
        strategy_name = strategy or self.default_strategy

        # Check if we have a cached result
        cache_key = f"{strategy_name}:{text}"
        if cache_key in self.text_cache:
            print(f"Using cached result for '{cache_key}'")
            return self.text_cache[cache_key]

        # Get the strategy from the registry
        if strategy_name not in self.strategies:
            msg = f"Unknown strategy: {strategy_name}"
            raise ValueError(msg)

        # Apply the strategy
        result = self.strategies[strategy_name](text)

        # Cache the result
        self.text_cache[cache_key] = result

        return result


# Add methods to the registry
TextProcessor.strategies.update_from_object(TextProcessor)

# Create a processor
processor = TextProcessor(default_strategy="capitalize")

# Process text with different strategies
text = "hello world"
print(f"Default strategy: {processor.process(text)}")
print(f"Uppercase strategy: {processor.process(text, 'uppercase')}")
print(f"Lowercase strategy: {processor.process(text, 'lowercase')}")
print(f"Reverse strategy: {processor.process(text, 'reverse')}")

# Process the same text again to demonstrate caching
print("\nProcessing again with default strategy:")
print(f"Result: {processor.process(text)}")

Default strategy: Hello World
Uppercase strategy: HELLO WORLD
Lowercase strategy: hello world
Reverse strategy: dlrow olleh

Processing again with default strategy:
Using cached result for 'capitalize:hello world'
Result: Hello World


### Method Registry as a Plugin System

Another advanced use case is implementing a plugin system where plugins can register methods that will be automatically bound to instances:

In [16]:
# Create a base plugin class
class Plugin:
    """Base class for plugins."""

    def __init__(self, name) -> None:
        self.name = name

    def get_name(self):
        return self.name


# Create a plugin manager class
class PluginManager:
    # Create a method registry for plugin methods
    plugin_methods = MethodRegistry()

    def __init__(self) -> None:
        self.plugins = {}

    def register_plugin(self, plugin) -> None:
        """Register a plugin and its methods."""
        self.plugins[plugin.name] = plugin

        # Get all methods from the plugin
        for name in dir(plugin):
            # Skip private methods and non-callable attributes
            if name.startswith("_") or not callable(getattr(plugin, name)):
                continue

            # Add the method to the registry with a prefixed name
            method = getattr(plugin, name)
            method_name = f"{plugin.name}_{name}"
            self.plugin_methods.data[method_name] = method.__func__ if hasattr(method, "__func__") else method

    def execute_method(self, method_name, *args, **kwargs):
        """Execute a plugin method by name."""
        # Check if the method exists
        if method_name not in self.plugin_methods:
            msg = f"Unknown method: {method_name}"
            raise ValueError(msg)

        # Get the plugin name from the method name
        plugin_name = method_name.split("_")[0]

        # Get the plugin instance
        if plugin_name not in self.plugins:
            msg = f"Plugin not registered: {plugin_name}"
            raise ValueError(msg)

        plugin = self.plugins[plugin_name]

        # Create a bound registry for the plugin
        bound_registry = self.plugin_methods.__get__(plugin)

        # Execute the method
        return bound_registry[method_name](*args, **kwargs)

    def list_methods(self):
        """List all available plugin methods."""
        return list(self.plugin_methods.keys())


# Create some plugins
class TextPlugin(Plugin):
    def __init__(self, name="text") -> None:
        super().__init__(name)

    def format(self, text, prefix="", suffix="") -> str:
        return f"{prefix}{text}{suffix}"

    def count_words(self, text):
        return len(text.split())


class MathPlugin(Plugin):
    def __init__(self, name="math") -> None:
        super().__init__(name)

    def add(self, a, b):
        return a + b

    def multiply(self, a, b):
        return a * b


# Create a plugin manager
manager = PluginManager()

# Register plugins
text_plugin = TextPlugin()
math_plugin = MathPlugin()
manager.register_plugin(text_plugin)
manager.register_plugin(math_plugin)

# List available methods
print("Available methods:", manager.list_methods())

# Execute plugin methods
print(f"\nText plugin format: {manager.execute_method('text_format', 'Hello World', prefix='>> ', suffix=' <<')}")
print(f"Text plugin word count: {manager.execute_method('text_count_words', 'Hello World')}")
print(f"Math plugin add: {manager.execute_method('math_add', 10, 5)}")
print(f"Math plugin multiply: {manager.execute_method('math_multiply', 10, 5)}")

Available methods: ['text_count_words', 'text_format', 'text_get_name', 'math_add', 'math_get_name', 'math_multiply']

Text plugin format: >> Hello World <<
Text plugin word count: 2
Math plugin add: 15
Math plugin multiply: 50


### Using BoundMethodRegistry Directly

While `MethodRegistry` is typically used through the descriptor protocol, you can also create and use `BoundMethodRegistry` directly for more control:

In [17]:
from baseobjects.functions.methodregistry import BoundMethodRegistry, BaseMethodRegistry


# Create a class with methods
class DataAnalyzer:
    def __init__(self, name) -> None:
        self.name = name

    def analyze_text(self, text) -> str:
        return f"{self.name} analyzed text: {len(text)} characters, {len(text.split())} words"

    def analyze_numbers(self, numbers) -> str:
        return f"{self.name} analyzed numbers: min={min(numbers)}, max={max(numbers)}, avg={sum(numbers) / len(numbers):.2f}"


# Create instances
analyzer1 = DataAnalyzer("Basic Analyzer")
analyzer2 = DataAnalyzer("Advanced Analyzer")

# Create a base method registry
base_registry = BaseMethodRegistry()
base_registry.update_from_object(analyzer1)

# Create bound registries for different instances
bound_registry1 = BoundMethodRegistry(registry=base_registry, instance=analyzer1)
bound_registry2 = BoundMethodRegistry(registry=base_registry, instance=analyzer2)

# Use the bound registries
print(f"Bound to analyzer1: {bound_registry1['analyze_text']('Hello World')}")
print(f"Bound to analyzer2: {bound_registry2['analyze_text']('Hello World')}")

print(f"\nBound to analyzer1: {bound_registry1['analyze_numbers']([1, 2, 3, 4, 5])}")
print(f"Bound to analyzer2: {bound_registry2['analyze_numbers']([1, 2, 3, 4, 5])}")

Bound to analyzer1: Basic Analyzer analyzed text: 11 characters, 2 words
Bound to analyzer2: Advanced Analyzer analyzed text: 11 characters, 2 words

Bound to analyzer1: Basic Analyzer analyzed numbers: min=1, max=5, avg=3.00
Bound to analyzer2: Advanced Analyzer analyzed numbers: min=1, max=5, avg=3.00


This approach gives you more control over when and how methods are bound to instances, which can be useful in more complex scenarios where you need to manage the binding process explicitly.

## Examples

### Example 1: Command Pattern with Method Binding

The Command pattern is a behavioral design pattern that turns a request into a stand-alone object containing all information about the request. Using `MethodRegistry`, we can implement a command pattern that automatically binds commands to their respective handlers:

In [18]:
# Create a command handler class
class CommandHandler:
    # Create a registry for commands
    commands = MethodRegistry()

    def __init__(self, name) -> None:
        self.name = name
        self.history = []

    def save_command(self, filename, content):
        """Command to save content to a file."""
        print(f"[{self.name}] Saving content to {filename}...")
        # In a real application, this would actually save to a file
        result = f"Saved {len(content)} characters to {filename}"
        self.history.append(("save", filename, len(content)))
        return result

    def load_command(self, filename):
        """Command to load content from a file."""
        print(f"[{self.name}] Loading content from {filename}...")
        # In a real application, this would actually load from a file
        result = f"Loaded content from {filename}"
        self.history.append(("load", filename))
        return result

    def process_command(self, text):
        """Command to process text."""
        print(f"[{self.name}] Processing text: {text[:20]}...")
        result = text.upper()
        self.history.append(("process", len(text)))
        return result

    def show_history(self) -> None:
        """Show the command execution history."""
        print(f"\n[{self.name}] Command History:")
        for i, cmd in enumerate(self.history, 1):
            print(f"{i}. {cmd}")


# Register commands
CommandHandler.commands.update_from_object(CommandHandler)

# Create command handlers
handler1 = CommandHandler("FileHandler")
handler2 = CommandHandler("BackupHandler")

# Execute commands on handler1
print(handler1.commands["save_command"]("document.txt", "This is some content to save."))
print(handler1.commands["load_command"]("document.txt"))
print(handler1.commands["process_command"]("Hello, world!"))

# Execute commands on handler2
print("\n" + handler2.commands["save_command"]("backup.txt", "Backup content"))
print(handler2.commands["load_command"]("backup.txt"))

# Show command history for both handlers
handler1.show_history()
handler2.show_history()

[FileHandler] Saving content to document.txt...
Saved 29 characters to document.txt
[FileHandler] Loading content from document.txt...
Loaded content from document.txt
[FileHandler] Processing text: Hello, world!...
HELLO, WORLD!
[BackupHandler] Saving content to backup.txt...

Saved 14 characters to backup.txt
[BackupHandler] Loading content from backup.txt...
Loaded content from backup.txt

[FileHandler] Command History:
1. ('save', 'document.txt', 29)
2. ('load', 'document.txt')
3. ('process', 13)

[BackupHandler] Command History:
1. ('save', 'backup.txt', 14)
2. ('load', 'backup.txt')


In this example, we've implemented a command pattern using `MethodRegistry`. The `CommandHandler` class has a `commands` registry that holds all the command methods. When we access the registry through an instance (e.g., `handler1.commands`), the methods are automatically bound to that instance, allowing us to execute commands in the context of the specific handler.

### Example 2: Event System with Method Registry

Another practical example is implementing an event system where event handlers are automatically bound to their respective objects:

In [19]:
# Create an event system
class EventSystem:
    def __init__(self) -> None:
        self.listeners = {}

    def register_listener(self, event_type, listener) -> None:
        """Register a listener for an event type."""
        if event_type not in self.listeners:
            self.listeners[event_type] = []
        self.listeners[event_type].append(listener)

    def unregister_listener(self, event_type, listener) -> None:
        """Unregister a listener for an event type."""
        if event_type in self.listeners and listener in self.listeners[event_type]:
            self.listeners[event_type].remove(listener)

    def dispatch_event(self, event_type, *args, **kwargs):
        """Dispatch an event to all registered listeners."""
        if event_type not in self.listeners:
            return None

        results = []
        for listener in self.listeners[event_type]:
            # Call the appropriate handler method on the listener
            if hasattr(listener, "handlers") and event_type in listener.handlers:
                result = listener.handlers[event_type](*args, **kwargs)
                results.append(result)

        return results


# Create a listener class
class EventListener:
    # Create a registry for event handlers
    handlers = MethodRegistry()

    def __init__(self, name) -> None:
        self.name = name
        self.events_received = []

    def on_message(self, message) -> str:
        """Handle message events."""
        print(f"[{self.name}] Received message: {message}")
        self.events_received.append(("message", message))
        return f"Message processed by {self.name}"

    def on_user_login(self, user_id, username) -> str:
        """Handle user login events."""
        print(f"[{self.name}] User login: {username} (ID: {user_id})")
        self.events_received.append(("login", user_id, username))
        return f"Login processed by {self.name}"

    def on_user_logout(self, user_id) -> str:
        """Handle user logout events."""
        print(f"[{self.name}] User logout: ID {user_id}")
        self.events_received.append(("logout", user_id))
        return f"Logout processed by {self.name}"

    def show_events(self) -> None:
        """Show all events received by this listener."""
        print(f"\n[{self.name}] Events Received:")
        for i, event in enumerate(self.events_received, 1):
            print(f"{i}. {event}")


# Register event handlers
EventListener.handlers.update_from_object(EventListener)

# Create an event system
event_system = EventSystem()

# Create listeners
listener1 = EventListener("SecurityMonitor")
listener2 = EventListener("ActivityLogger")

# Register listeners for different events
event_system.register_listener("message", listener1)
event_system.register_listener("login", listener1)
event_system.register_listener("logout", listener1)

event_system.register_listener("message", listener2)
event_system.register_listener("login", listener2)

# Dispatch events
print("Dispatching message event:")
event_system.dispatch_event("message", "System maintenance scheduled")

print("\nDispatching login event:")
event_system.dispatch_event("login", 12345, "admin")

print("\nDispatching logout event:")
event_system.dispatch_event("logout", 12345)

# Show events received by each listener
listener1.show_events()
listener2.show_events()

Dispatching message event:

Dispatching login event:

Dispatching logout event:

[SecurityMonitor] Events Received:

[ActivityLogger] Events Received:


In this example, we've implemented an event system using `MethodRegistry`. The `EventListener` class has a `handlers` registry that holds all the event handler methods. When an event is dispatched, the appropriate handler method is called on each registered listener, with the method automatically bound to the listener instance.

## API Highlights

The `MethodRegistry` module provides several classes for managing methods and their binding to instances:

### MethodRegistry

The main class that implements the descriptor protocol for automatic method binding:

- **Inheritance**: Extends `BaseMethodRegistry`
- **Key Method**: `__get__(self, instance, owner)` - Returns a `BoundMethodRegistry` when accessed through an instance

### BaseMethodRegistry

The abstract base class for method registries:

- **Inheritance**: Extends `FunctionRegistry` and `BaseReducible`
- **Constructor**: `__init__(methods=None, object_=None, objects=None, ...)`
- **Key Methods**:
  - `construct(methods=None, object_=None, objects=None, ...)` - Constructor method
  - `__func__` property - Gets/sets the underlying FunctionRegistry

### BoundMethodRegistry

A registry that is bound to a specific instance:

- **Inheritance**: Extends `BaseMethodRegistry`
- **Constructor**: `__init__(registry=None, instance=None, owner=None, ...)`
- **Key Methods**:
  - `construct(registry=None, instance=None, owner=None, ...)` - Constructor method
  - `__self__` property - Gets/sets the bound instance
  - `__getstate__` and `__setstate__` - Handle pickling and unpickling

### Key Features

- **Dictionary-like Interface**: Inherits from `BaseDict` through `FunctionRegistry`, providing familiar dictionary operations
- **Method Storage**: Stores callable methods and maintains their binding context
- **Descriptor Protocol**: Implements `__get__` to automatically bind methods to instances
- **Object Integration**: Methods to extract callable attributes from objects
- **Serialization Support**: Proper handling of pickling and unpickling through `BaseReducible`

For the full API documentation, refer to the baseobjects documentation.

## Troubleshooting / FAQs

### Q: What's the difference between FunctionRegistry and MethodRegistry?

A: While both store callable objects, `FunctionRegistry` stores functions that need to be called with an explicit instance parameter, while `MethodRegistry` automatically binds methods to instances when accessed through the descriptor protocol, eliminating the need for an explicit instance parameter.

### Q: How do I access methods in a MethodRegistry?

A: There are two main ways:
1. Through the descriptor protocol: `instance.registry['method_name']()`
2. By creating a bound registry: `registry.__get__(instance)['method_name']()`

### Q: Can I use MethodRegistry with static methods or class methods?

A: Yes, `MethodRegistry` can store static methods and class methods. Static methods will be accessible as regular functions, while class methods will be bound to the class when accessed through the class.

### Q: How do I handle methods with the same name from different objects?

A: When adding methods from multiple objects, methods with the same name will overwrite each other. To keep methods with the same name from different objects, you can use a naming convention or prefix/suffix:

```python
# Add methods with prefixes
for name in dir(obj1):
    attr = getattr(obj1, name)
    if callable(attr):
        registry.data[f"obj1_{name}"] = attr.__func__ if hasattr(attr, "__func__") else attr

for name in dir(obj2):
    attr = getattr(obj2, name)
    if callable(attr):
        registry.data[f"obj2_{name}"] = attr.__func__ if hasattr(attr, "__func__") else attr
```

### Q: Why am I getting a TypeError when calling a method from the registry?

A: This could happen if you're trying to call a method from a `MethodRegistry` without proper binding. Make sure you're accessing the registry through an instance (using the descriptor protocol) or creating a `BoundMethodRegistry` with `registry.__get__(instance)`.

### Q: Can I use MethodRegistry with inheritance?

A: Yes, `MethodRegistry` works well with inheritance. When a subclass inherits a class attribute that is a `MethodRegistry`, the registry will be shared among all instances of the class and its subclasses, but methods will be bound to the specific instance through which they are accessed.

## Conclusion and Next Steps

In this tutorial, we've explored the `MethodRegistry` class from the baseobjects package. We've learned how it extends the functionality of `FunctionRegistry` by adding automatic method binding through the descriptor protocol.

### Key Takeaways

- `MethodRegistry` provides a convenient way to organize and manage callable methods
- It automatically binds methods to instances when accessed through the descriptor protocol
- It can be used as a class attribute to create shared method registries
- It integrates with `BaseReducible` for proper serialization support
- It can be extended to create custom registries with additional functionality
- It's useful for implementing design patterns like Command and Strategy with automatic binding

### Next Steps

- Explore other function-related classes in the baseobjects package, such as:
  - `FunctionRegistry` for managing functions without automatic binding
  - `BaseDecorator` for creating function decorators
  - `SingleKwargDispatch` for dispatching functions based on keyword arguments
- Create your own custom method registries by subclassing `MethodRegistry`
- Combine `MethodRegistry` with other components of the baseobjects package
- Check out the examples directory for more examples of using `MethodRegistry`

For more information, refer to the baseobjects documentation and examples.